In [0]:
try:
    spark.conf.set(
    "fs.azure.account.key.accstoragedeproject.dfs.core.windows.net",
    dbutils.secrets.get(scope="key-vault-scope", key="storage-key")
    )
except:
    raise Exception("Nie można uzyskać dostępu do Key Vault.")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType

In [0]:
storage_name = 'accstoragedeproject'
bronze_path = f'abfss://bronze@{storage_name}.dfs.core.windows.net/'
silver_path = f'abfss://silver@{storage_name}.dfs.core.windows.net/'

order_list_schema = StructType([
    StructField('Order ID', StringType(), nullable=False),
    StructField('Order Date', StringType()),
    StructField('CustomerName', StringType()),
    StructField('State', StringType()),
    StructField('City', StringType())
])
order_details_schema = StructType([
    StructField('Order ID', StringType(), nullable=False),
    StructField('Amount', DoubleType()),
    StructField('Profit', DoubleType()),
    StructField('Quantity', IntegerType()),
    StructField('Category', StringType()),
    StructField('Sub-Category', StringType())
])

df_order_list = spark.read.csv(bronze_path + f'List%20of%20Orders.csv', header=True, schema=order_list_schema, mode='PERMISSIVE')
df_order_details = spark.read.csv(bronze_path + f'Order%20Details.csv', header=True, schema=order_details_schema, mode='PERMISSIVE')

In [0]:
df_order_list_cleaned = (df_order_list

    .withColumnRenamed('Order ID', 'Order_ID')
    .withColumnRenamed('Order Date', 'Order_Date')
    .withColumnRenamed('CustomerName', 'Customer_Name')

    .filter((F.col('Order_ID').isNotNull()) & (F.length(F.col('Order_ID')) > 2))
    .dropDuplicates(['Order_ID'])

    .withColumn('Order_Date', F.to_date(F.col('Order_Date'), 'dd-MM-yyyy'))

    .withColumns({'Order_ID': F.trim(F.col('Order_ID')), 'Customer_Name': F.trim(F.col('Customer_Name')), 'State': F.trim(F.col('State')), 'City': F.trim(F.col('City'))})
)

In [0]:
df_order_details_cleaned = (df_order_details

    .withColumnRenamed('Order ID', 'Order_ID_FK')
    .withColumnRenamed('Sub-Category', 'Sub_Category')

    .filter((F.col('Order_ID_FK').isNotNull()) & (F.length(F.col('Order_ID_FK')) > 2))

    .withColumns({'Order_ID_FK': F.trim(F.col('Order_ID_FK')), 'Category': F.trim(F.col('Category')), 'Sub_Category': F.trim(F.col('Sub_Category'))})
)

In [0]:
df_silver_final = df_order_list_cleaned.join(
    df_order_details_cleaned,
    df_order_list_cleaned.Order_ID == df_order_details_cleaned.Order_ID_FK,
    how='inner'
    ).drop('Order_ID_FK')

df_silver_final = df_silver_final.withColumn('silver_load_timestamp', F.current_timestamp())

In [0]:
df_silver_final.write.format('delta').mode('overwrite').option("overwriteSchema", "true").save(silver_path + 'cleaned_joined_orders')